In [3]:
# =============================================
# APP WEB
# =============================================

code = '''
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(page_title="Predicción MnO2", layout="centered")
st.title("🔬 Predicción de Concentración de MnO2")
st.subheader("Modelo XGBoost - Electroobtención de Zinc")

# Cargar modelos
@st.cache_resource
def load_models():
    modelo = joblib.load('modelo_xgboost_mn.pkl')
    scaler = joblib.load('scaler_mn.pkl')
    detector = joblib.load('detector_outliers_mn.pkl')
    features = joblib.load('features_mn.pkl')
    return modelo, scaler, detector, features

modelo, scaler, detector_outliers, features = load_models()

# --- Entradas del usuario ---
st.sidebar.header("Parámetros Operativos")

temp = st.sidebar.slider("Temperatura de EA (°C)", 31.0, 46.0, 38.0, step=0.1)
acidez = st.sidebar.slider("Acidez (%)", 161.0, 189.0, 175.0, step=0.1)
ph = st.sidebar.slider("pH del electrolito", 0.8, 2.2, 1.5, step=0.01)
densidad = st.sidebar.slider("Densidad (g/L)", 1285.0, 1305.0, 1295.0, step=0.1)
zn_ea = st.sidebar.slider("Zn en EA (g/L)", 38.0, 48.0, 42.0, step=0.1)
peso = st.sidebar.slider("Peso del depósito (kg)", 75.0, 90.0, 82.0, step=0.1)

# Valores fijos o por defecto
zn_pura = st.sidebar.number_input("Zn Sol Pura (g/L)", value=150.0)
fe = st.sidebar.number_input("Fe (mg/L)", value=3.0)
sb = st.sidebar.number_input("Sb (mg/L)", value=0.03)
cu = st.sidebar.number_input("Cu (mg/L)", value=0.1)

# Botón para predecir
if st.button("Predecir Concentración de MnO2", type="primary"):
    input_dict = {
        'T de EA (°C)': temp,
        'Zn Sol Pura (g/L)': zn_pura,
        'Fe (mg/L)': fe,
        'Sb (mg/L)': sb,
        'Cu (mg/L)': cu,
        'Densidad (g/L)': densidad,
        'Zn EA (g/L)': zn_ea,
        'Acidez (%)': acidez,
        'Horas de depósito': 48.0,
        'Peso Depósito (kg)': peso,
        'pH_electrolito': ph
    }
    
    df_input = pd.DataFrame([input_dict])
    df_input = df_input[features]
    
    input_scaled = scaler.transform(df_input)
    
    prediccion = modelo.predict(input_scaled)[0]
    es_outlier = detector_outliers.predict(input_scaled)[0]
    
    st.success(f"**Concentración de MnO2 predicha: {prediccion:.2f} g/L**")
    
    if es_outlier == 1:
        st.info("Parámetros dentro del rango de entrenamiento. Predicción confiable.")
    else:
        st.warning("⚠️ Parámetros fuera del rango habitual. La predicción puede no ser confiable.")
        
    st.write("### Parámetros ingresados:")
    st.json(input_dict)
'''

# Guardar el archivo app.py
with open("app.py", "w", encoding="utf-8") as f:
    f.write(code)

print("Archivo 'app.py' creado correctamente en la carpeta actual.")
print("Ahora puedes ejecutar la app desde Anaconda Prompt con:")
print("streamlit run app.py")

Archivo 'app.py' creado correctamente en la carpeta actual.
Ahora puedes ejecutar la app desde Anaconda Prompt con:
streamlit run app.py
